# Student Social Media Addiction & Mental Health Impact — A Behavioral Analytics Study

* Social media is no longer just a communication tool for students — it has become a lifestyle. A student today wakes up, checks Instagram before breakfast, attends class, unlocks their phone 150+ times through the day, and sleeps with YouTube playing. What that daily pattern quietly does to their sleep, their study hours, their stress, and their mental health — that is exactly what this project investigates.

* This end-to-end analytics project analyzes behavioral data from 5,000 students across academic levels, countries, and platforms to uncover the real relationship between social media habits and student well-being.

In [18]:
from sqlalchemy import create_engine , text
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
engine = create_engine(
    f"mysql+mysqlconnector://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}/{os.environ['DB_NAME']}"
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tables = [row[0] for row in result]

print("Total Tables:", len(tables))
print("Table Names:")
for table in tables:
    print("-", table)

Total Tables: 1
Table Names:
- student_social_media_and_mental_health


In [7]:
for table in tables:
       query = text(f"SELECT COUNT(*) FROM {table}")
       df = pd.read_sql_query(query, engine)
       print(f"{table}", df.iloc[0,0])
       df = pd.read_sql(f"SELECT * FROM {table}", engine)
       display(df.head(5))

student_social_media_and_mental_health 5000


,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134.0,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73.0,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166.0,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220.0,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237.0,1.0,1.1,5.0,Very High,4.4


In [ ]:
print(f"{df.info()}")
print("-"* 40)
print(f"null check : \n{df.isnull().sum()}")
print("-"* 40)
print(f"duplicate data : {df.duplicated().sum()}")
print("-"* 40)
print(f"data shape : \n{df.shape}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      5000 non-null   int64  
 1   Gender                   5000 non-null   object 
 2   Country                  5000 non-null   object 
 3   Academic_Level           5000 non-null   object 
 4   Most_Used_Platform       5000 non-null   object 
 5   Purpose_Of_Use           5000 non-null   object 
 6   Avg_Daily_Usage_Hours    5000 non-null   float64
 7   Daily_Unlocks            5000 non-null   float64
 8   Study_Hours              5000 non-null   float64
 9   Physical_Activity_Hours  5000 non-null   float64
 10  Sleep_Hours_Per_Night    5000 non-null   float64
 11  Stress_Level             5000 non-null   object 
 12  Mental_Health_Score      5000 non-null   float64
dtypes: float64(6), int64(1), object(6)
memory usage: 507.9+ KB
None
--------------

In [13]:
# show oroginal data with duplicate
df[df.duplicated(keep=False)]

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
886,19,Female,Other,Undergraduate,LinkedIn,Education,6.1,202.0,1.0,2.1,5.8,High,4.1
1870,21,Male,USA,Undergraduate,Snapchat,Entertainment,3.9,126.0,5.1,2.6,7.1,Low,7.5
2405,19,Female,Other,Undergraduate,LinkedIn,Education,6.1,202.0,1.0,2.1,5.8,High,4.1
2463,21,Male,USA,Undergraduate,Snapchat,Entertainment,3.9,126.0,5.1,2.6,7.1,Low,7.5


In [14]:
df = df.drop_duplicates()

In [15]:
print(f"data shape after drop duplicate : \n{df.shape}")

data shape after drop duplicate : 
(4998, 13)


In [19]:
# remove whitespace from object columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.strip()

In [20]:
# check unique values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    vals = df[col].unique()
    if(len(vals) <=30):
        print(f"\n{col} ({len(vals)} unique): {sorted(vals)}")


Gender (2 unique): ['Female', 'Male']

Academic_Level (3 unique): ['Graduate', 'High School', 'Undergraduate']

Most_Used_Platform (12 unique): ['Facebook', 'Instagram', 'KakaoTalk', 'LINE', 'LinkedIn', 'Snapchat', 'TikTok', 'Twitter', 'VKontakte', 'WeChat', 'WhatsApp', 'YouTube']

Purpose_Of_Use (4 unique): ['Education', 'Entertainment', 'Networking', 'News']

Stress_Level (4 unique): ['High', 'Low', 'Medium', 'Very High']


In [21]:
df.columns

Index(['Age', 'Gender', 'Country', 'Academic_Level', 'Most_Used_Platform',
       'Purpose_Of_Use', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
       'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night',
       'Stress_Level', 'Mental_Health_Score'],
      dtype='object')

In [22]:
df.to_sql(name="clean_dataset",
          con=engine , 
          if_exists="replace" , 
          index=False,
          chunksize=500)
print("clean dataset save in mysql")

clean dataset save in mysql


In [23]:
df_verify = pd.read_sql("SELECT * FROM clean_dataset LIMIT 5", engine)
display(df_verify)

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134.0,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73.0,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166.0,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220.0,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237.0,1.0,1.1,5.0,Very High,4.4


In [24]:
original_count = pd.read_sql("SELECT COUNT(*) AS cnt FROM clean_dataset", engine)
print(f"✓ Rows in MySQL  : {original_count['cnt'][0]}")
print(f"✓ Rows in df     : {len(df)}")
print(f"✓ Match          : {original_count['cnt'][0] == len(df)}")

✓ Rows in MySQL  : 4998
✓ Rows in df     : 4998
✓ Match          : True
